# Data loading

MSGPACK to GeoParquet.

In [1]:
from ship_routing.app.routing import RoutingResult, RoutingLog
from load_tuning_results import (
    load_results_raw,
    load_result_for_key,
    load_results,
    get_journey_params_df,
    get_hyper_params_df,
    get_runtime_df,
    get_elite_df,
    get_forcing_df,
    get_diversity_df,
)

In [2]:
from pathlib import Path

import pandas as pd
import geopandas as gpd

In [3]:
import warnings

warnings.filterwarnings("ignore")

In [4]:
data_files = sorted(Path("../results/").glob("results_*.msgpack"))
print(len(data_files))

7


In [5]:
results = load_results(data_files)
len(results)

records:   0%|          | 0/23165 [00:00<?, ?it/s]

23165

In [6]:
# Load individual DataFrames
df_hyper = get_hyper_params_df(results)
df_journey = get_journey_params_df(results)
df_runtime = get_runtime_df(results)
df_elite = get_elite_df(results)
df_forcing = get_forcing_df(results)
df_diversity = get_diversity_df(results)

elite:   0%|          | 0/23165 [00:00<?, ?it/s]

diversity:   0%|          | 0/23165 [00:00<?, ?it/s]

diversity:   0%|          | 0/23165 [00:00<?, ?it/s]

In [7]:
# Merge: params (1-to-1) → runtime (1-to-1) → elite (1-to-many with left join)
df_merged = (
    df_hyper.merge(df_journey, left_index=True, right_index=True, how="inner")
    .merge(df_runtime, left_index=True, right_index=True, how="inner")
    .merge(df_elite, left_index=True, right_index=True, how="left")
    .merge(df_forcing, left_index=True, right_index=True, how="left")
    .merge(df_diversity, left_index=True, right_index=True, how="left")
)

df_merged

,hyper_population_size,hyper_random_seed,hyper_selection_acceptance_rate_warmup,hyper_mutation_width_fraction_warmup,hyper_mutation_displacement_fraction_warmup,hyper_generations,hyper_offspring_size,hyper_crossover_rounds,hyper_selection_quantile,hyper_selection_acceptance_rate,...,elite_length_relative,elite_cost_absolute,elite_cost_relative,geometry,forcing_scenario_name,forcing_currents_path,forcing_waves_path,forcing_winds_path,diversity_cost_q75_25_rms,diversity_cost_q50_00_rms
filename,,,,,,,,,,,,,,,,,,,,,
result:0166:seed304346679:hashe07f51fa,64,304346679,0.3,0.99,0.25,1,16,0,0.10,0.25,...,1.094635,1.327261e+13,0.868933,"LINESTRING (-80.5 29.999999999999986, -79.4512...",baseline,data_large/cmems_mod_glo_phy_my_0.083deg_P1D-m...,data_large/cmems_mod_glo_wav_my_0.2deg_PT3H-i_...,data_large/cmems_obs-wind_glo_phy_my_l4_0.125d...,2.697679e+11,1.428100e+12
result:0166:seed304346679:hashe07f51fa,64,304346679,0.3,0.99,0.25,1,16,0,0.10,0.25,...,1.094635,1.327261e+13,0.868933,"LINESTRING (-80.5 29.999999999999986, -79.4512...",baseline,data_large/cmems_mod_glo_phy_my_0.083deg_P1D-m...,data_large/cmems_mod_glo_wav_my_0.2deg_PT3H-i_...,data_large/cmems_obs-wind_glo_phy_my_l4_0.125d...,2.697679e+11,1.428100e+12
result:0403:seed3552650797:hashb483dc3f,64,3552650797,0.3,0.99,0.25,1,64,0,0.10,0.25,...,1.113073,1.123693e+13,0.867480,LINESTRING (-11.000000000000002 50.00000000000...,baseline,data_large/cmems_mod_glo_phy_my_0.083deg_P1D-m...,data_large/cmems_mod_glo_wav_my_0.2deg_PT3H-i_...,data_large/cmems_obs-wind_glo_phy_my_l4_0.125d...,2.268287e+11,1.217095e+12
result:0403:seed3552650797:hashb483dc3f,64,3552650797,0.3,0.99,0.25,1,64,0,0.10,0.25,...,1.113073,1.123693e+13,0.867480,LINESTRING (-11.000000000000002 50.00000000000...,baseline,data_large/cmems_mod_glo_phy_my_0.083deg_P1D-m...,data_large/cmems_mod_glo_wav_my_0.2deg_PT3H-i_...,data_large/cmems_obs-wind_glo_phy_my_l4_0.125d...,2.268287e+11,1.217095e+12
result:0425:seed2987091933:hash316daa41,64,2987091933,0.3,0.99,0.25,1,16,0,0.25,0.00,...,1.001842,6.966852e+12,0.975147,LINESTRING (-11.000000000000002 50.00000000000...,baseline,data_large/cmems_mod_glo_phy_my_0.083deg_P1D-m...,data_large/cmems_mod_glo_wav_my_0.2deg_PT3H-i_...,data_large/cmems_obs-wind_glo_phy_my_l4_0.125d...,2.739303e+10,1.775624e+11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
result:3436:seed1758643021:hashaf644f49,256,1758643021,0.3,0.99,0.25,4,128,2,0.10,0.00,...,1.124445,5.440552e+12,0.637992,"LINESTRING (-11 50.00000000000001, -12.2306813...",no_winds,data_large/cmems_mod_glo_phy_my_0.083deg_P1D-m...,data_large/cmems_mod_glo_wav_my_0.2deg_PT3H-i_...,NaN,6.917543e+11,1.202694e+12
result:1592:seed1441152222:hashac1a0f5c,256,1441152222,0.3,0.99,0.25,4,256,2,0.10,0.25,...,1.002334,3.707139e+12,0.875200,"LINESTRING (-80.5 29.99999999999999, -79.65241...",no_winds,data_large/cmems_mod_glo_phy_my_0.083deg_P1D-m...,data_large/cmems_mod_glo_wav_my_0.2deg_PT3H-i_...,NaN,7.462919e+10,1.880100e+11
result:1592:seed1441152222:hashac1a0f5c,256,1441152222,0.3,0.99,0.25,4,256,2,0.10,0.25,...,1.002334,3.707139e+12,0.875200,"LINESTRING (-80.5 29.99999999999999, -79.65241...",no_winds,data_large/cmems_mod_glo_phy_my_0.083deg_P1D-m...,data_large/cmems_mod_glo_wav_my_0.2deg_PT3H-i_...,NaN,7.462919e+10,1.880100e+11


In [8]:
df_merged.columns

Index(['hyper_population_size', 'hyper_random_seed',
       'hyper_selection_acceptance_rate_warmup',
       'hyper_mutation_width_fraction_warmup',
       'hyper_mutation_displacement_fraction_warmup', 'hyper_generations',
       'hyper_offspring_size', 'hyper_crossover_rounds',
       'hyper_selection_quantile', 'hyper_selection_acceptance_rate',
       'hyper_mutation_width_fraction', 'hyper_mutation_displacement_fraction',
       'hyper_mutation_iterations', 'hyper_crossover_strategy',
       'hyper_hazard_penalty_multiplier', 'hyper_num_elites',
       'hyper_gd_iterations', 'hyper_learning_rate_time',
       'hyper_learning_rate_space', 'hyper_time_increment',
       'hyper_distance_increment', 'hyper_enable_adaptation',
       'hyper_target_relative_improvement', 'hyper_adaptation_scale_W',
       'hyper_adaptation_scale_D', 'hyper_W_min', 'hyper_W_max', 'hyper_D_min',
       'hyper_D_max', 'hyper_num_workers', 'hyper_executor_type',
       'journey_name', 'journey_lon_waypoints

In [9]:
gpd.GeoDataFrame(df_merged).to_parquet("../results/results_prelim.geoparquet")